# Evaluación de la recuperación semántica híbrida

La comparación utiliza un subconjunto anotado de Matterport3D/R2R. Los edificios de entrenamiento y validación seleccionan pesos y rechazo; los edificios de prueba se evalúan una sola vez. La configuración solo se congela cuando los tres splits y su ground truth están disponibles.


In [ ]:
from pathlib import Path
import sys

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_navigation_ws" / "src").is_dir())
sys.path.insert(0, str(repo / "experiments" / "shared"))

import os
import subprocess
import time
import numpy as np
import pandas as pd

from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from semantic_evaluation.core.dataset_adapters import load_dataset
from semantic_evaluation.core.offline_dataset import load_queries

ctx = bootstrap_offline()
config = ctx["config"]
spec = next(item for item in ctx["dataset_specs"] if item.dataset_id == "matterport3d")
dataset = load_dataset(spec, ctx["repo_root"])
queries = load_queries(str(resolve_repo_path(ctx["repo_root"], spec.queries_file)))
split_by_scan = dataset.metadata.get("building_split_by_scan", {})
split_counts = pd.Series(split_by_scan).value_counts().to_dict() if split_by_scan else {}
print({"available": not dataset.skipped, "nodes": len(dataset.nodes),
       "queries": len(queries), "buildings_by_split": split_counts,
       "reason": dataset.skip_reason})


## Preparación de representaciones


In [ ]:
from semantic_evaluation.core import EmbeddingCache
from semantic_evaluation.core.offline_encoding import (
    detect_objects, encode_observations, embed_texts, infer_dataset_relations)
from semantic_vision_core import SemanticVisionPipeline

ready = not dataset.skipped and bool(dataset.nodes) and bool(queries) and {
    "train", "validation", "test"}.issubset(set(split_by_scan.values()))
pipeline = None
if ready:
    checkpoint_value = config["models"]["yolo"]["checkpoint"]
    checkpoint = None if "${" in checkpoint_value else resolve_repo_path(ctx["repo_root"], checkpoint_value)
    if checkpoint is None or not checkpoint.is_file():
        ready = False
        print("Falta YOLO_CHECKPOINT local; calibración omitida.")
if ready:
    siglip = config["models"]["siglip"]
    cache = EmbeddingCache(str(resolve_repo_path(
        ctx["repo_root"], config["paths"]["cache_root"]) / "matterport_hybrid"))
    try:
        pipeline = SemanticVisionPipeline(
            retrieval_mode="siglip_yolo", siglip_model_id=siglip["model_id"],
            yolo_model_path=str(checkpoint),
            yolo_confidence_threshold=config["models"]["yolo"]["confidence_threshold"],
            device=ctx["device"], processor_fast=siglip["processor_fast"],
            local_files_only=siglip.get("local_files_only", False))
        encode_observations(dataset.nodes, pipeline, cache, siglip["model_id"])
        detect_objects(dataset.nodes, pipeline, cache, str(checkpoint),
                       config["models"]["yolo"]["confidence_threshold"], True,
                       siglip["model_id"])
        infer_dataset_relations(dataset.nodes)
    except (ImportError, OSError, RuntimeError) as error:
        ready = False
        print("Modelos no disponibles:", error)
if not ready:
    print("Se requieren Matterport3D/R2R, anotaciones, splits por edificio y modelos locales.")


## Comparación y selección en entrenamiento/validación


In [ ]:
from dataclasses import replace
from itertools import product
from semantic_evaluation.core.experiment_runner import prepare_queries, run_method
from semantic_evaluation.core.retrieval_metrics import summarize, results_to_rows
from semantic_navigation_core import HybridWeights, MultiviewConfig
from semantic_navigation_core.retrieval import (
    METHOD_SINGLE_VIEW_SIGLIP, METHOD_MULTIVIEW_SIGLIP,
    METHOD_SIGLIP_WITH_OBJECTS, METHOD_SIGLIP_WITH_OBJECTS_AND_RELATIONS,
    METHOD_HYBRID_SEMANTIC_RETRIEVAL, RetrievalConfig)

all_results = []
selected_weights = None
selected_threshold = None
calibration_table = pd.DataFrame()
validation_queries = []
if ready and pipeline is not None:
    prepared = prepare_queries(queries, dataset)
    embeddings = embed_texts([item.query.text for item in prepared], pipeline, cache,
                             config["models"]["siglip"]["model_id"])
    for item, embedding in zip(prepared, embeddings):
        item.embedding = embedding
    mv_data = config["retrieval"]["multiview"]
    mv_config = MultiviewConfig(**mv_data)
    def core_weights(name):
        values = config["retrieval"]["weights"][name]
        return HybridWeights(alpha=values["global_similarity"],
                             beta=values["object_match"], gamma=values["crop_similarity"],
                             delta=values["relation_match"], epsilon=values["room_match"])
    node_split = {node.node_id: split_by_scan.get(node.node_id.split(":", 1)[0])
                  for node in dataset.nodes}
    # Candidate nodes remain inside their building split; no building crosses splits.
    split_datasets = {
        split: replace(dataset, nodes=[node for node in dataset.nodes
                                      if node_split[node.node_id] == split])
        for split in ("train", "validation", "test")}
    def queries_for_split(split):
        return [item for item in prepared if (
            item.query.metadata.get("split") == split if item.query.is_negative
            else any(node_split.get(node_id) == split for node_id in item.valid_node_ids))]
    train_queries = queries_for_split("train")
    validation_queries = queries_for_split("validation")
    candidate_rows = []
    for alpha, beta in product((0.40, 0.50, 0.60), (0.05, 0.10, 0.15)):
        gamma = 0.75 - alpha - beta
        if gamma < 0:
            continue
        weights = HybridWeights(alpha=alpha, beta=beta, gamma=gamma,
                                delta=0.10, epsilon=0.15)
        train_result = run_method(
            "hybrid_semantic_retrieval", train_queries, split_datasets["train"],
            RetrievalConfig(method=METHOD_HYBRID_SEMANTIC_RETRIEVAL,
                            multiview=mv_config, weights=weights),
            config["retrieval"]["rejection"]["initial_threshold"])
        train_mrr = summarize(train_result)[0]["mean_reciprocal_rank"] if train_result else np.nan
        candidate_rows.append({"weights": weights, "train_mrr": train_mrr})
    finalists = sorted(candidate_rows, key=lambda row: row["train_mrr"], reverse=True)[:5]
    for row in finalists:
        validation_result = run_method(
            "hybrid_semantic_retrieval", validation_queries,
            split_datasets["validation"],
            RetrievalConfig(method=METHOD_HYBRID_SEMANTIC_RETRIEVAL,
                            multiview=mv_config, weights=row["weights"]),
            config["retrieval"]["rejection"]["initial_threshold"])
        row["validation_mrr"] = (
            summarize(validation_result)[0]["mean_reciprocal_rank"]
            if validation_result else np.nan)
        row["validation_results"] = validation_result
    if finalists and np.isfinite([row["validation_mrr"] for row in finalists]).any():
        selected = max(finalists, key=lambda row: row["validation_mrr"])
        selected_weights = selected["weights"]
        from semantic_evaluation.core.evaluation_statistics import calibrate_rejection_threshold
        positives = [item.scores[0] for item in selected["validation_results"]
                     if not item.is_negative and item.scores]
        negatives = [item.scores[0] for item in selected["validation_results"]
                     if item.is_negative and item.scores]
        try:
            selected_threshold, threshold_metrics = calibrate_rejection_threshold(
                positives, negatives)
        except ValueError as error:
            print("Umbral no calibrado:", error)
        calibration_table = pd.DataFrame([
            {"alpha": row["weights"].alpha, "beta": row["weights"].beta,
             "gamma": row["weights"].gamma, "delta": row["weights"].delta,
             "epsilon": row["weights"].epsilon, "train_mrr": row["train_mrr"],
             "validation_mrr": row["validation_mrr"]}
            for row in finalists])
display(calibration_table)


## Evaluación única en edificios de prueba


In [ ]:
if ready and pipeline is not None and selected_weights is not None and selected_threshold is not None:
    test_queries = queries_for_split("test")
    method_configs = {
        "single_view_siglip": RetrievalConfig(method=METHOD_SINGLE_VIEW_SIGLIP),
        "multiview_siglip": RetrievalConfig(method=METHOD_MULTIVIEW_SIGLIP, multiview=mv_config),
        "siglip_with_objects": RetrievalConfig(method=METHOD_SIGLIP_WITH_OBJECTS,
                                                multiview=mv_config,
                                                weights=core_weights("siglip_with_objects")),
        "siglip_with_objects_and_relations": RetrievalConfig(
            method=METHOD_SIGLIP_WITH_OBJECTS_AND_RELATIONS, multiview=mv_config,
            weights=core_weights("siglip_with_objects_and_relations")),
        "hybrid_semantic_retrieval": RetrievalConfig(
            method=METHOD_HYBRID_SEMANTIC_RETRIEVAL, multiview=mv_config,
            weights=selected_weights),
    }
    for label, method_config in method_configs.items():
        all_results.extend(run_method(label, test_queries, split_datasets["test"],
                                      method_config, selected_threshold))
cases = pd.DataFrame(results_to_rows(all_results))
summary = pd.DataFrame(summarize(all_results, ("method", "query_type")))
display(summary)


## Configuración congelada


In [ ]:
from reproducibility import freeze_retrieval_config

frozen = None
if ready and selected_weights is not None and selected_threshold is not None and not cases.empty:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ctx["repo_root"],
                            capture_output=True, text=True, check=False).stdout.strip()
    frozen_payload = {
        "siglip_checkpoint": config["models"]["siglip"]["model_id"],
        "yolo_checkpoint": str(checkpoint),
        "preprocessing": {"processor_fast": config["models"]["siglip"]["processor_fast"]},
        "multiview": config["retrieval"]["multiview"],
        "hybrid_weights": selected_weights.__dict__,
        "rejection_threshold": selected_threshold,
        "category_mappings": {},
        "relation_extractor_version": "semantic_navigation_core.relations.v1",
        "git_commit": commit,
    }
    output = resolve_repo_path(ctx["repo_root"], config["paths"]["results_root"])
    frozen = freeze_retrieval_config(
        output / "frozen_retrieval_config.yaml", frozen_payload,
        {"validated_offline": True, "n_validation_queries": len(validation_queries),
         "simulation_sources": [], "building_splits": split_counts})
    print("Configuración congelada:", frozen)
else:
    print("No se genera configuración congelada: faltan datos o calibración válida.")


## Latencia, memoria, errores e interpretación


In [ ]:
from semantic_evaluation.core import HardwareSampler

if cases.empty:
    print("No hay resultados que interpretar.")
else:
    display(cases.groupby("method")[["recall_at_1", "reciprocal_rank",
                                      "retrieval_latency_ms"]].mean(numeric_only=True))
    failures = cases.loc[(cases["is_negative"] == False) & (cases["recall_at_1"] == False)]
    display(failures[["query_id", "method", "predicted_node_id",
                      "valid_node_ids", "rank_first_valid"]].head(20))
    hardware = HardwareSampler().sample()
    print({"cpu_percent": hardware.cpu_percent, "ram_used_mb": hardware.ram_used_mb})
    best = summary.dropna(subset=["recall_at_1"]).sort_values("recall_at_1", ascending=False).iloc[0]
    print(f"La mayor Recall@1 calculada fue {best['recall_at_1']:.3f} para {best['method']} "
          f"en consultas {best['query_type']}.")
